# Project 2: Wildfire Mapping
### SDS210 — Programming with Spatial Data
**University of Zurich · FS 2026**

---

## Project overview

This project is organised as a pipeline of four notebooks. Each one writes an intermediate file that the next reads, so they must be executed **in this order**:

1. **`data_retrieval.ipynb`** — Retrieves wildfire detection data from the NASA FIRMS API (VIIRS S-NPP, Near Real-Time) and saves it as a CSV in `data/raw/`.
2. **`data_cleaning.ipynb`** — Cleans the raw records (missing values, dates/times, confidence labels) and writes a tidy dataset to `data/processed/`.
3. **`exploratory_analysis.ipynb`** — Answers four spatial and statistical questions about global fire activity.
4. **`interactive_web_map.ipynb`** — Visualises the results on Folium web maps exported to `outputs/maps/`.

> **Run order matters:** each notebook reads the output of the previous step. Running them out of order will cause `FileNotFoundError` or stale results. Use `Kernel → Restart & Run All` before moving on to the next notebook.

### Research questions

| # | Question |
|---|----------|
| Q1 | Where are the most severe active wildfires (highest Fire Radiative Power)? |
| Q2 | How does thermal intensity (FRP) vary across continents? |
| Q3 | Are wildfires more commonly detected during daytime or nighttime passes, and does this differ by region? |
| Q4 | How does detection confidence relate to fire intensity (FRP)? |

### Data source

**NASA FIRMS** (Fire Information for Resource Management System)  
Product: **VIIRS S-NPP NRT** (Near Real-Time, 375 m resolution)  
API: https://firms.modaps.eosdis.nasa.gov/api/  
Time window: last 5 days, global extent

> **API key:** A free MAP_KEY is needed for the API. See section 2.1 below for how to provide it. Without a key, the notebook falls back to the cached CSV in `data/raw/`.

---
## 1. Setup

All imports for this notebook are grouped in the cell below. The downstream notebooks have their own setup cells.

In [1]:
# --- Standard library ---
import os
import warnings
from io import StringIO
from pathlib import Path
from datetime import datetime, timezone

# --- Data ---
import pandas as pd

# --- HTTP requests (FIRMS API) ---
import requests

# Suppress minor deprecation warnings for a clean output
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Setup complete. Timestamp: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}")

Setup complete. Timestamp: 2026-05-22 09:45 UTC


---
## 2. Data Access — NASA FIRMS API

### 2.1 Configuration

The MAP_KEY is read from one of two places, in order:

1. the `FIRMS_MAP_KEY` environment variable, or
2. a local file `secrets/firms_map_key.txt` (containing just the key on one line).

The `secrets/` folder is listed in `.gitignore` so the key never reaches the repository. If neither source provides a key, the notebook falls back to the cached CSV in `data/raw/`. Register for a free key at https://firms.modaps.eosdis.nasa.gov/api/map_key/. The same key works for both the global and US/Canada FIRMS services; the transaction limit is 5000 requests per 10-minute window.

In [2]:
# ============================================================
#  USER CONFIGURATION
# ============================================================

# --- Project paths (relative — work on any machine) ---
# Detect the project root whether the notebook is opened from
# notebooks/ or directly from the project root.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebook", "notebooks"} else cwd

DATA_DIR   = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CACHE_FILE = DATA_DIR / "firms_viirs_global_5d.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- NASA FIRMS API key ---
# Source 1: environment variable
MAP_KEY = os.environ.get("FIRMS_MAP_KEY", "")

# Source 2: local secret file (gitignored)
key_file = PROJECT_ROOT / "secrets" / "firms_map_key.txt"
if not MAP_KEY and key_file.exists():
    MAP_KEY = key_file.read_text().strip()

# --- Request parameters ---
DAYS   = 5                  # 1–10
SOURCE = "VIIRS_SNPP_NRT"   # FIRMS product identifier
AREA   = "world"            # or a bbox like "5.9,45.8,10.5,47.9" for ~Switzerland

print(f"Project root    : {PROJECT_ROOT}")
print(f"Data directory  : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Cache file      : {CACHE_FILE}")
print(f"MAP_KEY loaded  : {'yes' if MAP_KEY else 'no (will use cached CSV)'}")

Project root    : /Users/davidleu/Desktop/sds210-wildfire-mapping-project
Data directory  : /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw
Output directory: /Users/davidleu/Desktop/sds210-wildfire-mapping-project/outputs
Cache file      : /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw/firms_viirs_global_5d.csv
MAP_KEY loaded  : yes


### 2.2 Fetch function

A single function handles both the API call and the local cache. By default it is **cache-first**: if `data/raw/firms_viirs_global_5d.csv` already exists, it is reused and no API call is made — this protects the snapshot the analysis is based on and avoids burning API transactions. Pass `force_refresh=True` to deliberately re-download from the API and overwrite the cache.

In [3]:
def fetch_firms_data(
    map_key: str,
    source: str,
    days: int,
    cache_path: Path,
    force_refresh: bool = False,
) -> pd.DataFrame:
    """
    Retrieve wildfire detection records from the NASA FIRMS API.

    Default behaviour is cache-first: if cache_path exists, the cached
    CSV is returned and no API call is made. To pull fresh data from
    the API and overwrite the cache, pass force_refresh=True.

    Parameters
    ----------
    map_key       : NASA FIRMS MAP_KEY string (empty -> cache only).
    source        : FIRMS product name, e.g. 'VIIRS_SNPP_NRT'.
    days          : Number of days of data to request (1-10).
    cache_path    : Path to the local CSV cache file.
    force_refresh : If True, ignore the cache and hit the API.

    Returns
    -------
    pd.DataFrame with raw FIRMS records.

    Raises
    ------
    FileNotFoundError if no cache exists and no API call is possible.
    """
    # --- 1. Cache-first: use the local CSV unless explicitly refreshing ---
    if cache_path.exists() and not force_refresh:
        raw_df = pd.read_csv(cache_path)
        print(f"Using cached data — {len(raw_df):,} records from {cache_path}")
        print("  (set force_refresh=True to re-download from the API)")
        return raw_df

    # --- 2. Live API path (only if no cache or force_refresh=True) ---
    if map_key.strip():
        base_url = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
        url = f"{base_url}/{map_key}/{source}/world/{days}"

        print(f"Connecting to FIRMS API … ({source}, {days} days, global)")
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            raw_df = pd.read_csv(StringIO(response.text))

            # Only write the cache on a successful download
            raw_df.to_csv(cache_path, index=False)
            print(f"Success — {len(raw_df):,} records retrieved. Cached to {cache_path}")
            return raw_df

        except requests.exceptions.RequestException as exc:
            print(f"API request failed ({exc}).")
            if cache_path.exists():
                print("Falling back to cached file …")
                return pd.read_csv(cache_path)
            raise

    # --- 3. No cache, no key ---
    raise FileNotFoundError(
        f"No cached file found at '{cache_path}' and no MAP_KEY was provided.\n"
        "Either register for a free key at "
        "https://firms.modaps.eosdis.nasa.gov/api/map_key/, "
        "or place a FIRMS CSV at the cache path manually."
    )

In [4]:
# --- Execute the data retrieval ---
# Default: cache-first. The cached CSV in data/raw/ is reused if it exists,
# so this cell is safe to re-run without spending API transactions or
# overwriting the snapshot the analysis is based on.
#
# Set force_refresh=True only when you deliberately want fresh data from
# the FIRMS API (this will overwrite the cache).
raw_df = fetch_firms_data(
    map_key       = MAP_KEY,
    source        = SOURCE,
    days          = DAYS,
    cache_path    = CACHE_FILE,
    force_refresh = False,
)

print(f"\nRows loaded: {len(raw_df):,}")
print(f"Columns    : {list(raw_df.columns)}")
raw_df.head(3)

Using cached data — 133,061 records from /Users/davidleu/Desktop/sds210-wildfire-mapping-project/data/raw/firms_viirs_global_5d.csv
  (set force_refresh=True to re-download from the API)

Rows loaded: 133,061
Columns    : ['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight']


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-10.38626,34.97153,311.74,0.78,0.78,2026-05-07,1,N,VIIRS,n,2.0NRT,285.90,2.92,N
1,-8.20365,32.76895,302.43,0.52,0.67,2026-05-07,1,N,VIIRS,n,2.0NRT,288.00,0.88,N
2,-6.85891,12.39353,319.00,0.37,0.58,2026-05-07,1,N,VIIRS,n,2.0NRT,287.29,8.19,N


---
## Reproducibility notes

To keep this project reproducible:

- All input data lives in `data/raw/`, all derived data in `data/processed/`, all final outputs in `outputs/`.
- All paths are relative — the notebook detects the project root regardless of whether it is opened from `notebooks/` or from the project root itself.
- Package requirements are listed in `environment.yml` at the project root.
- The cached CSV in `data/raw/` is the exact snapshot the submitted analysis is based on. The retrieval notebook reads it by default; re-running with `force_refresh=True` will fetch current data from FIRMS and produce different results, because FIRMS only serves the last 5–10 days.
- The notebook runs top-to-bottom without manual intervention after `Kernel → Restart & Run All`.